# CareLoop Clinics: Appointment No-Show Risk Scoring
**Objective:** Train and validate supervised classification models to flag high-risk appointments the evening before the visit, optimizing the decision threshold for front-desk recall.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

### 1. Train/Test Partition & Preprocessing Pipeline
We isolate 20% of records as a holdout test set (`random_state=42`) and construct a `ColumnTransformer` so feature scaling and one-hot encoding happen strictly inside cross-validation folds without data leakage.

In [ ]:
data = pd.read_csv("appointments_clean.csv")
appt_ids = data["appointment_id"].copy()
data["reminder_sent"] = (data["reminder_sent"] == "Yes").astype(int)

NUMERIC_FEATURES = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent"]
CATEGORICAL_FEATURES = ["booking_channel", "visit_type"]

X = data[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = data["no_show"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train Rows: {len(X_train)} | Test Rows: {len(X_test)} | Test No-Shows: {y_test.sum()}")
print(f"Majority-Class Baseline Accuracy: {(y_test == 0).mean():.4f}")

# Preprocessors
lr_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(drop=None, sparse_output=False), CATEGORICAL_FEATURES),
])

tree_preprocessor = ColumnTransformer([
    ("num", "passthrough", NUMERIC_FEATURES),
    ("cat", OneHotEncoder(drop=None, sparse_output=False), CATEGORICAL_FEATURES),
])

### 2. Model Benchmarking & Interpretability Analysis
We evaluate three model families—Logistic Regression, Depth-Constrained Decision Trees, and Random Forests—and inspect the odds-ratio impact of SMS/call reminders.

In [ ]:
lr_pipe = Pipeline([
    ("prep", lr_preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

dt_unbounded = Pipeline([
    ("prep", tree_preprocessor),
    ("model", DecisionTreeClassifier(random_state=42)),
])

dt_pruned = Pipeline([
    ("prep", tree_preprocessor),
    ("model", DecisionTreeClassifier(max_depth=3, random_state=42)),
])

rf_base = Pipeline([
    ("prep", tree_preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])

for pipe in [lr_pipe, dt_unbounded, dt_pruned, rf_base]:
    pipe.fit(X_train, y_train)

print(f"Unbounded Tree -> Train Acc: {accuracy_score(y_train, dt_unbounded.predict(X_train)):.4f} | Test Acc: {accuracy_score(y_test, dt_unbounded.predict(X_test)):.4f}")
print(f"Pruned Tree    -> Train Acc: {accuracy_score(y_train, dt_pruned.predict(X_train)):.4f} | Test Acc: {accuracy_score(y_test, dt_pruned.predict(X_test)):.4f}")

# Inspect Logistic Regression coefficients and true reminder odds ratio
feature_names = lr_pipe.named_steps["prep"].get_feature_names_out()
coefs = lr_pipe.named_steps["model"].coef_[0]
scaler_stds = lr_pipe.named_steps["prep"].named_transformers_["num"].scale_

reminder_pos = list(feature_names).index("num__reminder_sent")
reminder_std = scaler_stds[NUMERIC_FEATURES.index("reminder_sent")]
unscaled_reminder_coef = coefs[reminder_pos] / reminder_std

print(f"\nReminder Odds Ratio (per 1-unit change): {np.exp(unscaled_reminder_coef):.4f}")
print("Confusion Matrix (Logistic Regression @ 0.50):\n", confusion_matrix(y_test, lr_pipe.predict(X_test)))

### 3. 5-Fold Cross-Validation & Hyperparameter Tuning
We tune Random Forest hyperparameters via `GridSearchCV` and compare all candidate pipelines using 5-fold cross-validated ROC-AUC on the training split to select our champion model objectively.

In [ ]:
rf_grid = GridSearchCV(
    estimator=rf_base,
    param_grid={"model__n_estimators": [100, 200], "model__max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
)
rf_grid.fit(X_train, y_train)

candidates = {
    "Logistic Regression": (lr_pipe, cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()),
    "Decision Tree (Depth=3)": (dt_pruned, cross_val_score(dt_pruned, X_train, y_train, cv=5, scoring="roc_auc").mean()),
    "Random Forest (Base)": (rf_base, cross_val_score(rf_base, X_train, y_train, cv=5, scoring="roc_auc").mean()),
    "Random Forest (Tuned)": (rf_grid.best_estimator_, rf_grid.best_score_),
}

print("--- 5-Fold CV ROC-AUC Leaderboard ---")
for name, (_, score) in candidates.items():
    print(f"{name:<25}: {score:.4f}")

# Dynamically select the highest-scoring CV model
champion_name = max(candidates, key=lambda k: candidates[k][1])
champion_model = candidates[champion_name][0]
print(f"\nSelected Champion Model: {champion_name}")

### 4. Operating Threshold Calibration & Final Holdout Evaluation
Because a missed appointment leaves a costly empty slot while an outreach call takes minimal front-desk effort, we evaluate out-of-fold probabilities at thresholds `0.50` and `0.35` before scoring the test partition once.

In [ ]:
oof_probabilities = cross_val_predict(
    champion_model, X_train, y_train, cv=5, method="predict_proba"
)[:, 1]

for cutoff in [0.50, 0.35]:
    oof_flags = (oof_probabilities >= cutoff).astype(int)
    print(
        f"Threshold {cutoff:.2f} -> Precision: {precision_score(y_train, oof_flags, zero_division=0):.4f} | "
        f"Recall: {recall_score(y_train, oof_flags):.4f} | F1: {f1_score(y_train, oof_flags, zero_division=0):.4f} | "
        f"Flagged: {oof_flags.sum()}/{len(y_train)}"
    )

OPERATING_THRESHOLD = 0.35
champion_model.fit(X_train, y_train)

holdout_probs = champion_model.predict_proba(X_test)[:, 1]
holdout_flags = (holdout_probs >= OPERATING_THRESHOLD).astype(int)

print("\n--- Final Holdout Test Evaluation ---")
print(f"ROC-AUC   : {roc_auc_score(y_test, holdout_probs):.4f}")
print(f"Precision : {precision_score(y_test, holdout_flags, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, holdout_flags):.4f}")
print(f"Flagged   : {holdout_flags.sum()} / {len(X_test)}")

risk_output = pd.DataFrame({
    "appointment_id": appt_ids.loc[X_test.index],
    "no_show": y_test.values,
    "risk_prob": holdout_probs,
    "flag": holdout_flags,
})
risk_output.to_csv("test_risk_scores.csv", index=False)
print("\nExported test_risk_scores.csv successfully.")